# Runbook 1.2.c: Strengthening authentication: length/complexity, randomness, MFA

**Skill:** Mitigate Risk · **Maps to:** 1.2.C · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** Implement the server-side Time-based One-Time Password (TOTP) verifier that an authentication
backend would call to check a user's second factor. Given the shared secret, a submitted code,
and the current time, return True if the code is valid for the current 30-second window.

This is the algorithm defined in RFC 6238 (TOTP) / RFC 4226 (HOTP), implemented using only
Python's standard library `hmac` and `hashlib` modules.

**Lesson objective.** Explain why password length and randomness are the primary drivers of authentication strength, describe how Time-based One-Time Passwords work as a second factor, and implement a TOTP verifier using only the standard library.

### What comes in
- `secret` (`str`) -- the shared secret key, encoded as an **even-length lowercase hex string**
  (e.g. `"0a1b2c3d"` represents 4 bytes: `b'\x0a\x1b\x2c\x3d'`).
- `code` (`str`) -- the 6-digit code submitted by the user, zero-padded if necessary
  (e.g. `"007342"` or `"123456"`).
- `t` (`int`) -- the current Unix timestamp (seconds since epoch) at the moment of verification.

### What must come out
`True` if the computed TOTP code for the current 30-second step matches `code`; `False` otherwise.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
# Using key=bytes.fromhex("3132333435363738393031323334353637383930") (RFC 4226 test vector)
# At t=59 (step=1): expected code is "287082"
solve("3132333435363738393031323334353637383930", "287082", 59)   -> True
solve("3132333435363738393031323334353637383930", "287082", 60)   -> False  # step changed
solve("3132333435363738393031323334353637383930", "000000", 59)   -> False  # wrong code
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, stdlib only (`hmac`, `hashlib`).
- [ ] Accept only the CURRENT step (no window drift / no adjacent steps).
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(secret: str, code: str, t: int) -> bool
```

#### Input
- `secret` (`str`) -- even-length lowercase hex string encoding the TOTP shared secret key
  (e.g. `"0a1b2c3d"` = 4 bytes). Tests only use valid hex strings of 4-20 bytes.
- `code` (`str`) -- 6-character string of decimal digits, zero-padded (e.g. `"007342"`).
- `t` (`int`) -- current Unix timestamp (non-negative integer).

#### Output
`bool` -- `True` if `code` matches the TOTP for the 30-second step containing `t`; else `False`.

#### Algorithm (exact, deterministic -- all tests derive expected values from this spec)

```
key    = bytes.fromhex(secret)
step   = t // 30
msg    = step.to_bytes(8, byteorder='big')
digest = hmac.new(key, msg, hashlib.sha1).digest()   # 20 bytes

offset = digest[19] & 0x0f
p      = ((digest[offset] & 0x7f) << 24
         | digest[offset+1] << 16
         | digest[offset+2] << 8
         | digest[offset+3])
otp    = p % 1_000_000
expected = str(otp).zfill(6)

return code == expected
```

#### RFC 4226 test vector (for implementation validation)
Secret (ASCII "12345678901234567890" = 20 bytes, hex below):
```
secret = "3132333435363738393031323334353637383930"
```
Known correct codes per HOTP counter (step for TOTP = t // 30):

| step (t value) | expected code |
|----------------|---------------|
| 1  (t = 30-59)  | 287082        |
| 2  (t = 60-89)  | 359152        |
| 3  (t = 90-119) | 969429        |

These are derived from the RFC 4226 test vectors. the tests uses them to validate any
implementation.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network.
- Standard library only: `import hmac`, `import hashlib`.
- Only the CURRENT step is accepted (no window tolerance / no drift allowance).
- `code` is always exactly 6 characters. A code with fewer than 6 digits is already zero-padded
  by the caller.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

Fill in `solve` below. Run the cell (Shift+Enter) after each change.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
"""Starter code for 1.2.c. Student completes the TODO.
This file intentionally FAILS the tests until implemented."""


def solve(secret: str, code: str, t: int) -> bool:
    """Verify a 6-digit TOTP code against the shared secret at Unix timestamp t.

    Args:
        secret: even-length hex string encoding the shared key (e.g. "0a1b2c3d").
        code:   6-character zero-padded decimal string (e.g. "007342").
        t:      current Unix timestamp (int, seconds since epoch).
    Returns:
        True if code matches TOTP(secret, t // 30); False otherwise.

    Algorithm (from the Reference section below):
      1. key    = bytes.fromhex(secret)
      2. step   = t // 30
      3. msg    = step.to_bytes(8, byteorder='big')
      4. digest = hmac.new(key, msg, hashlib.sha1).digest()   # 20 bytes
      5. offset = digest[19] & 0x0f
      6. p      = ((digest[offset] & 0x7f) << 24) | (digest[offset+1] << 16)
                  | (digest[offset+2] << 8) | digest[offset+3]
      7. otp    = p % 1_000_000
      8. return code == str(otp).zfill(6)

    Import hmac and hashlib from the standard library only.
    """
    # TODO: implement the TOTP verifier -- See the problem statement above and the Reference section below
    raise NotImplementedError("implement solve() -- see the problem statement above")


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

RFC_SECRET = "3132333435363738393031323334353637383930"


def test_rfc_step1_correct():
    assert impl.solve(RFC_SECRET, "287082", 59) is True


def test_rfc_step2_correct():
    assert impl.solve(RFC_SECRET, "359152", 89) is True


def test_rfc_step3_correct():
    assert impl.solve(RFC_SECRET, "969429", 90) is True


def test_wrong_code_rejected():
    assert impl.solve(RFC_SECRET, "000000", 59) is False


def test_expired_code_rejected():
    assert impl.solve(RFC_SECRET, "287082", 60) is False


def test_same_step_different_timestamp():
    assert impl.solve(RFC_SECRET, "287082", 30) is True
    assert impl.solve(RFC_SECRET, "287082", 59) is True


def test_step_boundary():
    assert impl.solve(RFC_SECRET, "359152", 60) is True
    assert impl.solve(RFC_SECRET, "287082", 60) is False


def test_t_zero():
    result_true = impl.solve("0a1b2c3d", "497680", 0)
    result_false = impl.solve("0a1b2c3d", "000000", 0)
    assert result_true is True
    assert result_false is False


def test_different_key_same_time():
    assert impl.solve("0a1b2c3d", "497680", 0) is True
    assert impl.solve("deadbeef", "057490", 0) is True
    assert impl.solve("deadbeef", "497680", 0) is False


def test_replay_blocked():
    valid_code = "287082"
    assert impl.solve(RFC_SECRET, valid_code, 59) is True
    assert impl.solve(RFC_SECRET, valid_code, 60) is False
    assert impl.solve(RFC_SECRET, valid_code, 119) is False


def test_code_must_be_exact_string():
    assert impl.solve(RFC_SECRET, "28708", 59) is False
    assert impl.solve(RFC_SECRET, " 287082", 59) is False


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Call `solve` on an input you make up, and predict the answer first.

- [ ] Your prediction matched

## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it